In [ ]:
import csv
from google.colab import files # For file upload/download in Colab
import re # For more flexible option parsing

def main():
    """
    Main function to handle file upload, parsing (answer before question,
    including A, B, C, D options), CSV creation, and download.
    """
    print("Please upload your .txt file.")
    print("The expected format is a single line with the answer key (e.g., 'a', 'b'),")
    print("followed by a block of text for the question.")
    print("The question block can contain options like 'A. Text', 'B. Text', etc.")
    print("This pattern should repeat for each answer-question pair.")

    uploaded = files.upload()

    if not uploaded:
        print("\nNo file was uploaded. Exiting.")
        return

    file_name = list(uploaded.keys())[0]
    print(f"\nProcessing file: {file_name}")

    try:
        file_content = uploaded[file_name].decode('utf-8')
        print("File decoded successfully using UTF-8.")
    except UnicodeDecodeError:
        try:
            file_content = uploaded[file_name].decode('latin-1')
            print("File decoded successfully using latin-1 (fallback).")
        except UnicodeDecodeError:
            print("\nError: Could not decode the file.")
            print("Please ensure your file is encoded in UTF-8 or a similar common encoding.")
            return

    lines = file_content.splitlines()
    qa_data_list = [] # List to store dictionaries for each Q&A set
    current_question_lines_buffer = []
    current_answer_key = None

    # Regex to identify options like "A.", "A)", "a.", "a)" and capture the option letter and text
    # It looks for A, B, C, or D, case-insensitive, followed by a period or parenthesis,
    # then whitespace, and captures the rest of the line as the option text.
    option_regex = re.compile(r"^\s*([a-dA-D])[\.\)]\s*(.*)")

    def process_buffered_question_block(answer_key, question_lines):
        """
        Processes a block of question lines to extract the main question
        and individual options (A, B, C, D).
        """
        if not question_lines:
            return None

        main_question_parts = []
        options = {'A': None, 'B': None, 'C': None, 'D': None}

        for line in question_lines:
            match = option_regex.match(line)
            if match:
                option_letter = match.group(1).upper() # Normalize to uppercase A, B, C, D
                option_text = match.group(2).strip()
                if option_letter in options:
                    options[option_letter] = option_text
            else:
                main_question_parts.append(line)

        question_body = "\n".join(main_question_parts).strip()

        if not question_body and not any(options.values()): # If no actual question or options found
             print(f"Warning: Answer key '{answer_key}' was found, but no subsequent question text or options could be clearly identified.")
             return None

        return {
            'question': question_body,
            'option_a': options.get('A'),
            'option_b': options.get('B'),
            'option_c': options.get('C'),
            'option_d': options.get('D'),
            'answer': answer_key
        }

    for line_number, line_text in enumerate(lines, 1):
        stripped_text = line_text.strip()

        # Heuristic for an answer line:
        # - Exactly one character long after stripping.
        # - The character is an alphabet letter (case-insensitive).
        if len(stripped_text) == 1 and stripped_text.isalpha():
            # This line is an answer key.
            # Process the previously buffered question lines for the *previous* answer key.
            if current_answer_key is not None and current_question_lines_buffer:
                processed_data = process_buffered_question_block(current_answer_key, current_question_lines_buffer)
                if processed_data:
                    qa_data_list.append(processed_data)

            # Store the new answer key and reset the buffer for its question.
            current_answer_key = stripped_text # Keep original case for answer, or .lower()/.upper() if needed
            current_question_lines_buffer = []
        else:
            # This line is part of a question.
            if current_answer_key is not None: # Only collect question lines if an answer key has been identified
                current_question_lines_buffer.append(line_text)
            else:
                # Lines before the first answer key.
                if stripped_text:
                    print(f"Warning (Line {line_number}): Skipping line '{line_text[:70]}...' as it appears before the first answer key.")

    # After the loop, process the last buffered question for the last answer key
    if current_answer_key is not None and current_question_lines_buffer:
        processed_data = process_buffered_question_block(current_answer_key, current_question_lines_buffer)
        if processed_data:
            qa_data_list.append(processed_data)
    elif current_answer_key is not None and not current_question_lines_buffer : # An answer key was last, but no question followed
        print(f"Warning: Found answer key '{current_answer_key}' at the end of the file but no subsequent question text. This entry will be skipped.")


    if not qa_data_list:
        print("\nNo valid answer-question-options sets were extracted.")
        print("Please check your input file format matches the expected structure.")
        return

    output_csv_filename = 'parsed_questions_with_options.csv'
    # Define the column headers for the CSV. Answer key is at the end.
    fieldnames = ['question', 'option_a', 'option_b', 'option_c', 'option_d', 'answer']

    try:
        with open(output_csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for item_data in qa_data_list:
                writer.writerow(item_data)

        print(f"\nSuccessfully created '{output_csv_filename}' with {len(qa_data_list)} entries.")
        files.download(output_csv_filename)
        print(f"'{output_csv_filename}' has been prepared and offered for download.")

    except IOError:
        print(f"\nError: An IOError occurred while trying to write or download the CSV file '{output_csv_filename}'.")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")

if __name__ == '__main__':
    main()


Please upload your .txt file.
The expected format is a single line with the answer key (e.g., 'a', 'b'),
followed by a block of text for the question.
The question block can contain options like 'A. Text', 'B. Text', etc.
This pattern should repeat for each answer-question pair.


Saving Train.txt to Train.txt

Processing file: Train.txt
File decoded successfully using UTF-8.

Successfully created 'parsed_questions_with_options.csv' with 7376 entries.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'parsed_questions_with_options.csv' has been prepared and offered for download.
